# needle-fp4: one Colab run

Runtime > Change runtime type > **GPU**, then Runtime > **Run all**. Open this notebook from GitHub
(File > Open notebook > GitHub) rather than from the Drive clone: the setup cell resets the Drive clone to
match GitHub. If the repo is private, add a `GITHUB_TOKEN` secret first (see the setup cell).

The project asks two questions:
1. **Training:** a linear layer runs three matmuls (Fprop `Y = XW`, Dgrad `dX = dY Wᵀ`, Wgrad `dW = Xᵀ dY`).
   Which one breaks training when it runs in NVFP4, and does stochastic rounding (SR) fix it?
2. **Inference:** how much memory and GEMV time does real 4-bit weight storage save?

Only question 2 needs a GPU. Training uses fake quantization (values rounded to NVFP4, math in FP32), which the
tests below show is bit-identical on CPU and GPU, so the training grids were run on CPU and are in
`docs/results/` (MNIST and CIFAR-10, 3 seeds). This run therefore does, by default:
* build needle and run every test in `tests/fp4`, including the CUDA kernels;
* the GEMV benchmark (FP32 vs NVFP4, up to 8192 x 8192) and MNIST inference from packed 4-bit weights;
* zip the output (`fp4_results.zip`) to send back.

Set `RUN_TRAINING = True` to also repeat the MNIST and CIFAR-10 training grids on the GPU (much longer).

*AI-assisted: written with Claude (Anthropic) via Claude Code; reviewed and tested by the team.*


In [ ]:
BRANCH = "main"          # a dev/<name>-<topic> branch to try unmerged work
RUN_TRAINING = False     # True: also rerun the training grids on the GPU (long)
SEEDS = [0, 1, 2]
R = "results/colab"      # everything this run produces goes here


In [ ]:
import base64, subprocess
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/10714

# Private repo: create a GitHub token with read access to this repo, then in Colab click the key
# icon (Secrets) > Add new secret, name it GITHUB_TOKEN and allow notebook access. The token is
# passed to git as a header for each command and is never written to Drive.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
basic = base64.b64encode(f"x-access-token:{token}".encode()).decode() if token else ""
AUTH = f"-c http.extraheader='Authorization: Basic {basic}'" if token else ""
print("Using a GitHub token from Colab Secrets" if token else "No GITHUB_TOKEN secret: cloning as public")

!test -d fp4-needle || git {AUTH} clone https://github.com/shreya-v7/fp4-needle.git
%cd /content/drive/MyDrive/10714/fp4-needle
# Match GitHub exactly. Colab autosaves into the Drive copy of this notebook, which would
# otherwise make `git pull` refuse to update and leave old code in place.
# (Only tracked files are reset; results/, data/ and build/ are git-ignored and kept.)
!git {AUTH} fetch origin && git checkout -f {BRANCH} && git reset --hard origin/{BRANCH}
import subprocess
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
remote = subprocess.check_output(['git', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
assert head == remote, f"Checkout is at {head[:7]}, but origin/{BRANCH} is {remote[:7]}; see the git output above."
!git log --oneline -3

In [ ]:
!pip3 install -q pybind11
!make clean && make
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
sys.path.append('python'); sys.path.append('apps')
import needle as ndl
print('CUDA backend built:', ndl.cuda().enabled())
if not ndl.cuda().enabled():
    raise RuntimeError("No CUDA backend: Runtime > Change runtime type > GPU, then Runtime > Run all again.")

## 1. Tests
Every NVFP4 piece (CPU and CUDA kernels, autograd op, layers) is checked bit for bit against the NumPy reference.
The output is also saved to `results/colab/tests.txt`.


In [ ]:
import os
os.makedirs(R, exist_ok=True)
!python3 -m pytest tests/fp4 -q 2>&1 | tee {R}/tests.txt | tail -3


## 2. MNIST data


In [ ]:
import os, hashlib, shutil, urllib.request

# MNIST files and the MD5 of the copy our homeworks used (identical to the course repo).
MNIST = {"train-images-idx3-ubyte.gz": "0cb5796cd8e7c7bffef03edbdb01fac5",
         "train-labels-idx1-ubyte.gz": "23e0cdb926d142c2d470a3641a19d1ff",
         "t10k-images-idx3-ubyte.gz":  "3d4db374c734e4f92d346e063bf6e41a",
         "t10k-labels-idx1-ubyte.gz":  "0407fd84edb9a00bbad73c377ff95de9"}
os.makedirs("data", exist_ok=True)
for name, md5 in MNIST.items():
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        # Prefer a homework copy on Drive; otherwise download from the course's hw0 repo.
        local = [p for p in (f"../{hw}/data/{name}" for hw in ("hw2", "hw1", "hw0")) if os.path.exists(p)]
        if local:
            shutil.copy(local[0], dst)
        else:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/dlsyscourse/hw0/main/data/{name}", dst)
    assert hashlib.md5(open(dst, "rb").read()).hexdigest() == md5, f"{dst} is corrupted; delete it and rerun"
    print("ok", name)


## 3. Real 4-bit weights: memory and GEMV speed
Weights are stored packed (two 4-bit codes per byte, one E4M3 scale byte per 16 values, one FP32 scale per
matrix) and our CUDA kernel multiplies a vector directly from that form. This is the batch-1 text-generation
setting, where every layer is a matrix-vector product (GEMV) and speed is limited by reading the weights.


In [ ]:
!python3 benchmarks/bench_gemv.py --device cuda --sizes 1024 2048 4096 8192 --out {R}/bench_gemv.csv
from IPython.display import Image, Markdown, display
display(Image(f'{R}/bench_gemv.png'))
# MNIST MLP run from packed 4-bit weights: memory per layer, test accuracy, batch-1 latency
!python3 apps/fp4_inference.py --device cuda 2>&1 | tee {R}/inference.txt


## 4. (Optional) training grids on the GPU
Every configuration (FP32; Fprop; Fprop+Dgrad; all three; each backward matmul alone; with and without SR) on
MNIST (all 60,000 images, 3 epochs) and CIFAR-10 (20,000 images, 3 epochs), same settings as the CPU runs in
`docs/results/`. Only runs when `RUN_TRAINING = True`.


In [ ]:
import tarfile, urllib.request
if RUN_TRAINING:
    if not os.path.exists('data/cifar-10-batches-py'):
        urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', 'data/cifar-10-python.tar.gz')
        tarfile.open('data/cifar-10-python.tar.gz').extractall('data')
    for seed in SEEDS:
        !python3 apps/fp4_experiment.py --configs all --device cuda --epochs 3 --seed {seed} --out {R}/mnist_seed{seed}.csv
        !python3 apps/fp4_experiment.py --dataset cifar10 --configs all --device cuda --epochs 3 --train-subset 20000 --seed {seed} --out {R}/cifar_seed{seed}.csv
    !python3 apps/plot_fp4.py {R}/mnist_seed*.csv --out-dir {R}/mnist
    !python3 apps/plot_fp4.py {R}/cifar_seed*.csv --out-dir {R}/cifar --title "CIFAR-10 MLP" --prefix cifar
    display(Markdown(open(f'{R}/mnist/summary.md').read()))
    display(Markdown(open(f'{R}/cifar/summary.md').read()))


## 5. Send the results
Zips `results/colab` (test output, benchmark, inference log, and training CSVs if run). Download
`fp4_results.zip` from the Files panel on the left (or from `MyDrive/10714/fp4-needle/` on Drive) and send it.


In [ ]:
import shutil
shutil.make_archive('fp4_results', 'zip', '.', R)
print('Created', os.path.abspath('fp4_results.zip'), os.path.getsize('fp4_results.zip') // 1024, 'KB')
